# Spark Plan Viz — Example Gallery

Runnable examples for **every** optimization rule. Each section builds a DataFrame
whose physical plan triggers one (or more) findings, then visualizes / analyzes it.

> Requires: Python 3.11+, PySpark 3.5+ or 4.x, a local Java runtime (`JAVA_HOME`).
>
> Install editable: `uv sync --all-groups`


## Setup


In [ ]:
import os
import tempfile

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from spark_plan_viz import Severity, analyze_plan, visualize_plan

spark = (
    SparkSession.builder.master('local[2]')
    .appName('spark_plan_viz_examples')
    .config('spark.sql.shuffle.partitions', '5')
    .config('spark.ui.enabled', 'false')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate()
)

employees = spark.createDataFrame(
    [
        (1, 'Alice', 34, 'Engineering', 95000),
        (2, 'Bob', 45, 'Sales', 85000),
        (3, 'Cathy', 29, 'Engineering', 78000),
        (4, 'David', 38, 'Marketing', 72000),
        (5, 'Eve', 42, 'Sales', 88000),
        (6, 'Frank', 31, 'Engineering', 91000),
        (7, 'Grace', 27, 'Marketing', 67000),
        (8, 'Hank', 50, 'Sales', 102000),
    ],
    ['id', 'name', 'age', 'department', 'salary'],
)

departments = spark.createDataFrame(
    [
        ('Engineering', 'Tech', 'US'),
        ('Sales', 'Business', 'US'),
        ('Marketing', 'Business', 'EU'),
    ],
    ['dept_name', 'division', 'region'],
)

orders = spark.createDataFrame(
    [
        (1, 1, 100.0, '2024-01-15'),
        (2, 2, 250.0, '2024-01-16'),
        (3, 1, 75.0, '2024-02-01'),
        (4, 3, 300.0, '2024-02-10'),
        (5, 5, 180.0, '2024-03-01'),
        (6, 4, 90.0, '2024-03-15'),
    ],
    ['order_id', 'emp_id', 'amount', 'order_date'],
)


def show_findings(df, title: str = '') -> None:
    """Print analyzer findings and render the plan inline."""
    if title:
        print(f'\n=== {title} ===')
    suggestions = analyze_plan(df)
    if not suggestions:
        print('(no findings)')
    for s in suggestions:
        print(f'[{s.severity.value:7s}] {s.rule_id:28s} {s.title}')
    visualize_plan(df, notebook=True)


## Kitchen sink — multiple findings in one plan

A deliberately bad query that usually surfaces several rules at once
(`cross_join`, `expensive_collect`, and often shuffle-related infos).


In [ ]:
kitchen_sink = (
    employees.crossJoin(departments)
    .groupBy('division')
    .agg(F.collect_list('name').alias('all_names'))
)
show_findings(kitchen_sink, 'kitchen sink')


## Errors


### `cross_join` — Cartesian product

A cross join multiplies row counts. Almost always unintentional.


In [ ]:
# BAD
bad = employees.crossJoin(departments)
show_findings(bad, 'cross_join')

# FIX
good = employees.join(departments, employees.department == departments.dept_name)
show_findings(good, 'cross_join fixed')


### `nested_loop_join` — non-equality join

Range / inequality-only joins fall back to O(n*m) nested loops.


In [ ]:
# BAD
bad = employees.join(orders, employees.salary > orders.amount)
show_findings(bad, 'nested_loop_join')

# FIX — add an equality key
good = employees.join(
    orders,
    (employees.id == orders.emp_id) & (employees.salary > orders.amount),
)
show_findings(good, 'nested_loop_join fixed')


## Warnings


### `full_table_scan` — no pushed filters on Parquet/ORC/…


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'employees.parquet')
    employees.write.mode('overwrite').parquet(path)

    bad = spark.read.parquet(path).select('id', 'name')
    show_findings(bad, 'full_table_scan')

    good = spark.read.parquet(path).filter(F.col('age') > 30).select('id', 'name')
    show_findings(good, 'full_table_scan fixed')


### `empty_partition_filters` — partitioned table, no partition pruning

When data is written with `partitionBy`, filtering on the partition column
lets Spark skip directories. An empty `PartitionFilters` list means a full prune miss.


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'emp_part')
    employees.write.mode('overwrite').partitionBy('department').parquet(path)

    bad = spark.read.parquet(path)
    show_findings(bad, 'empty_partition_filters')

    good = spark.read.parquet(path).filter(F.col('department') == 'Sales')
    show_findings(good, 'empty_partition_filters fixed')


### `expand` — CUBE / ROLLUP / multiple COUNT DISTINCT

`Expand` multiplies each input row by the number of grouping sets.


In [ ]:
# BAD — 2-column cube → 4x expand
bad_cube = employees.cube('department', 'age').count()
show_findings(bad_cube, 'expand (cube)')

# BAD — multiple COUNT DISTINCT also inserts Expand
bad_cd = employees.groupBy('department').agg(
    F.countDistinct('id'),
    F.countDistinct('name'),
)
show_findings(bad_cd, 'expand (multi countDistinct)')

# BETTER — single aggregate / fewer grouping sets
good = employees.groupBy('department').agg(F.countDistinct('id').alias('n_ids'))
show_findings(good, 'expand avoided')


### `generate_explode` — array/map explosion

`explode` / `posexplode` / `inline` multiply rows by collection size.


In [ ]:
tagged = employees.withColumn(
    'tags', F.array(F.lit('a'), F.lit('b'), F.lit('c'))
)

# BAD
bad = tagged.select('id', F.explode('tags').alias('tag'))
show_findings(bad, 'generate_explode')

# BETTER — project narrow columns before explode
good = tagged.select('id', 'tags').select('id', F.explode('tags').alias('tag'))
show_findings(good, 'generate_explode (narrower)')


### `expensive_collect` — collect_list / collect_set


In [ ]:
bad = employees.groupBy('department').agg(F.collect_list('name').alias('names'))
show_findings(bad, 'expensive_collect')

good = employees.groupBy('department').agg(
    F.avg('salary').alias('avg_salary'),
    F.count('*').alias('headcount'),
)
show_findings(good, 'expensive_collect fixed')


### `window_without_partition` (+ often `single_partition_exchange`)

A window with only `orderBy` collapses work to one partition.


In [ ]:
# BAD
w_bad = Window.orderBy('salary')
bad = employees.withColumn('global_rank', F.row_number().over(w_bad))
show_findings(bad, 'window_without_partition')

# FIX
w_good = Window.partitionBy('department').orderBy('salary')
good = employees.withColumn('dept_rank', F.row_number().over(w_good))
show_findings(good, 'window_without_partition fixed')


### `python_udf` — row-wise Python UDFs


In [ ]:
@F.udf('string')
def upper_name(s):
    return s.upper() if s else None

bad = employees.select(upper_name('name').alias('upper_name'))
show_findings(bad, 'python_udf')

good = employees.select(F.upper('name').alias('upper_name'))
show_findings(good, 'python_udf fixed')


### `redundant_shuffle` — back-to-back exchanges


In [ ]:
# BAD — two shuffles in a row (sortWithinPartitions keeps the first exchange alive)
bad = (
    employees.repartition(10, 'department')
    .sortWithinPartitions('department')
    .repartition(5)
)
show_findings(bad, 'redundant_shuffle')

good = employees.repartition(10, 'department')
show_findings(good, 'redundant_shuffle fixed')


### `sort_before_shuffle` — sort destroyed by a following exchange

Sorting and then repartitioning throws away order.


In [ ]:
# BAD — Sort immediately under Exchange
bad = employees.orderBy('salary').repartition(4)
show_findings(bad, 'sort_before_shuffle')

# FIX — sort after the shuffle (or drop sort if order is not needed)
good = employees.repartition(4).sortWithinPartitions('salary')
show_findings(good, 'sort_before_shuffle fixed')


### `partition_count_low` / `partition_count_high`

Extreme shuffle partition counts hurt parallelism or scheduling.


In [ ]:
# BAD — force a 1-partition shuffle
bad_low = employees.repartition(1)
show_findings(bad_low, 'partition_count_low')

# BAD — enormous partition count
bad_high = employees.repartition(20000)
show_findings(bad_high, 'partition_count_high')

# BETTER
good = employees.repartition(8)
show_findings(good, 'partition_count ok')


### `non_columnar_no_pushdown` / `non_columnar_format` — CSV/JSON scans


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    csv_path = os.path.join(tmp, 'employees.csv')
    employees.write.mode('overwrite').csv(csv_path, header=True)

    # WARNING — row-based, no pushdown
    bad = spark.read.csv(csv_path, header=True)
    show_findings(bad, 'non_columnar_no_pushdown')

    # INFO path — row-based scan that may still record pushed filters
    maybe_info = spark.read.csv(csv_path, header=True).filter(F.col('age') > 30)
    show_findings(maybe_info, 'non_columnar_format (if pushdown present)')

    # FIX — columnar
    pq = os.path.join(tmp, 'employees.parquet')
    employees.write.mode('overwrite').parquet(pq)
    good = spark.read.parquet(pq).filter(F.col('age') > 30)
    show_findings(good, 'columnar fixed')


### `single_partition_exchange`

Global ops (unpartitioned windows, some aggregates) funnel work through one task.
See also `window_without_partition` above — the same plan often fires both.


In [ ]:
bad = employees.withColumn('rn', F.row_number().over(Window.orderBy('id')))
show_findings(bad, 'single_partition_exchange')


## Info


### `missing_broadcast_hint` — shuffle join that might broadcast


In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
try:
    bad = employees.join(departments, employees.department == departments.dept_name)
    show_findings(bad, 'missing_broadcast_hint')

    good = employees.join(
        F.broadcast(departments),
        employees.department == departments.dept_name,
    )
    show_findings(good, 'missing_broadcast_hint fixed')
finally:
    spark.conf.unset('spark.sql.autoBroadcastJoinThreshold')


### `coalesce` — round-robin `repartition(n)`


In [ ]:
bad = employees.repartition(2)
show_findings(bad, 'coalesce (round-robin repartition)')

good = employees.coalesce(2)
show_findings(good, 'coalesce fixed')


### `unnecessary_sort` — sort not consumed by an ordering-dependent op

Sorting before `dropDuplicates` can leave a Sort that is not required for the result.
(Spark often removes pure `orderBy → aggregate` sorts; this pattern still keeps the Sort.)


In [ ]:
# BAD — orderBy then dropDuplicates keeps a Sort that is not required
bad = employees.orderBy('salary').dropDuplicates(['department'])
show_findings(bad, 'unnecessary_sort')

# FIX — drop the unused orderBy
good = employees.dropDuplicates(['department'])
show_findings(good, 'unnecessary_sort fixed')


### `skew_join` — AQE skew handling is active (`isSkew=true`)

This rule is **informational**: it fires when Adaptive Query Execution has already
marked a join as skew-optimized. That flag appears on the **runtime / final** plan
after stages run with real skew — not on the static pre-execution plan of tiny toy data.

Below we illustrate the finding by attaching suggestions to a synthetic plan node
shaped like an AQE SortMergeJoin with `is_skew=true`.


In [ ]:
from spark_plan_viz._analyzer import _attach_suggestions

# Illustrative final-plan fragment (what you see after AQE skew split)
synthetic = {
    'name': 'SortMergeJoin',
    'description': 'SortMergeJoin [id#0], [emp_id#1], Inner, true',
    'type': 'join',
    'key_info': {'join_type': 'Inner', 'is_skew': True},
    'children': [],
    'output': [],
    'metrics': {},
    'suggestions': [],
}
findings = _attach_suggestions(synthetic)
for s in findings:
    print(f'[{s.severity.value:7s}] {s.rule_id:28s} {s.title}')
    print(f'  {s.message}')

print('Synthetic node suggestions:', synthetic['suggestions'])
print(
    'In real jobs: enable AQE, run an action on a skewed join, then re-explain /'
    ' visualize the executed plan (Spark UI SQL tab -> Final Plan).'
)


## New-rules combo — Expand + explode + partition miss

One workflow that stacks the rules added for the Spark 3.5/4 refresh.


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    part_path = os.path.join(tmp, 'emp_part')
    employees.write.mode('overwrite').partitionBy('department').parquet(part_path)

    # 1) Missed partition pruning
    scanned = spark.read.parquet(part_path)
    show_findings(scanned, 'combo: empty_partition_filters')

    # 2) Explode after tagging
    exploded = (
        scanned.withColumn('tags', F.array(F.lit('x'), F.lit('y')))
        .select('id', 'department', F.explode('tags').alias('tag'))
    )
    show_findings(exploded, 'combo: generate_explode')

    # 3) Cube over exploded rows -> Expand
    cubed = exploded.cube('department', 'tag').count()
    show_findings(cubed, 'combo: expand')


## Programmatic summary helper

Audit any DataFrame without opening the visualizer.


In [ ]:
result = (
    employees.crossJoin(departments)
    .join(orders, employees.id == orders.emp_id)
    .groupBy('division')
    .agg(F.collect_list('name').alias('all_names'))
)

suggestions = analyze_plan(result)
print(f'{len(suggestions)} finding(s)')
for s in suggestions:
    print(f'[{s.severity.value:7s}] {s.rule_id:28s} {s.title}')
    print(f'           {s.message}')

errors = [s for s in suggestions if s.severity == Severity.ERROR]
print(f'{len(errors)} error(s)')


## Complex realistic plan (visual polish)

Larger synthetic data to exercise joins, aggregates, and the HTML renderer.
Writes `notebooks/example.html` when the CWD is the repo root.


In [ ]:
products = spark.createDataFrame(
    [(i, f'Product_{i}', i * 10.5, f'Category_{i % 5}') for i in range(1, 201)],
    ['product_id', 'product_name', 'price', 'category'],
)
sales = spark.createDataFrame(
    [(i, i % 200 + 1, i * 2, f'2024-{(i % 12) + 1:02d}-01') for i in range(1, 1001)],
    ['sale_id', 'product_id', 'quantity', 'sale_date'],
)
customers = spark.createDataFrame(
    [(i, f'Customer_{i}', f'City_{i % 20}') for i in range(1, 101)],
    ['customer_id', 'customer_name', 'city'],
)

comprehensive = (
    sales.filter(F.col('quantity') > 1)
    .join(
        F.broadcast(products.filter(F.col('price') > 50)),
        on='product_id',
        how='inner',
    )
    .join(customers, sales.sale_id % 100 == customers.customer_id, how='left')
    .filter(F.col('category').isin(['Category_1', 'Category_2', 'Category_3']))
    .groupBy('category', 'city')
    .agg(
        F.sum(F.col('quantity') * F.col('price')).alias('total_revenue'),
        F.count('sale_id').alias('num_sales'),
        F.avg('price').alias('avg_price'),
    )
    .filter(F.col('total_revenue') > 1000)
    .orderBy(F.col('total_revenue').desc())
)

show_findings(comprehensive, 'comprehensive demo')
_ = visualize_plan(
    comprehensive,
    notebook=False,
    output_file='notebooks/example.html',
    open_browser=False,
)
print('Wrote notebooks/example.html')


In [ ]:
spark.stop()
